# 01 — Dicionário de dados
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:** lê os dados como vieram, guarda a camada bronze, descreve cada coluna (metadados) e
verifica a qualidade. Termina com o checklist das decisões de limpeza, tomadas pelo grupo em 10/11, que o notebook 02 aplica.

| Notebook | Papel | Seções |
|---|---|---|
| 00 — Documentação | O projeto de ponta a ponta, decisões e perguntas em aberto | — |
| `_comum` | Leitura dos dados, padrão dos gráficos, metadados e verificação de qualidade, usados por todos os notebooks | — |
| 01 — Dicionário de dados | O que é cada coluna, o que está errado e o que levar ao grupo | 1 a 4 |
| 02 — Preparação dos dados | Aplica as decisões do 01 e grava as tabelas limpas | — |
| 03 — Análise exploratória | Descritiva, associações, churn, vazamento e avaliação de cada variável | 5 a 10 |
| 04 — Variáveis derivadas e ABT | Variáveis derivadas, ABT e dicionário com a sugestão de cada coluna; grava a gold | — |
| 05 — Seleção de variáveis | Separa treino e teste, faz os testes estatísticos e chega às 41 variáveis do modelo; confere com os métodos da aula | — |
| 06 — Modelagem | Compara 6 modelos, escolhe o modelo e o limiar, publica as previsões e registra no MLflow | — |
| 07 — Segmentação | Agrupa os clientes por comportamento (K-means) e cruza com risco e valor | — |
| 08 — Entrega | Gera o `prediction` da entrega, confere o formato e resume os números para a apresentação | — |

A numeração das seções continua entre o 01 e o 03, para as referências cruzadas (por exemplo, "4.3" ou "7.3") valerem.

## 1. Leitura dos dados brutos

**O que faço:** carrego os dois arquivos exatamente como vieram, sem converter nada.

**Por que leio tudo como texto?**
- Se o pandas converter os tipos automaticamente, valores como `unknown` e `null` viram nulos (ou fazem a coluna
  inteira virar texto) **sem avisar**. Eu perderia a informação de **quais** problemas existem e **quantos**.
- O dicionário precisa ver o dado bruto para documentar a sujeira. A conversão de tipos só acontece na limpeza
  (notebook 02), de forma explícita e registrada.
- `keep_default_na=False` impede que o pandas transforme textos como `NA`, `null` ou vazio em nulo por conta própria.

**Por que uso o caminho completo?** O diretório de trabalho do Databricks nem sempre é a pasta do notebook.
Com o caminho absoluto, a leitura funciona de qualquer lugar.

In [ ]:
%run ./_comum

In [ ]:
display(hist_raw.head(3).T.reset_index(names="coluna"))   # transposto: uma linha por variável

> **Explicação**
> - **5.634 clientes no histórico e 1.409 na inferência.** As linhas físicas (5.912 e 1.461) são maiores porque
>   alguns comentários do `CustomerFeedback` ocupam mais de uma linha no arquivo. O pandas reconhece o texto entre
>   aspas e monta cada cliente corretamente. São esses números que o slide do hackathon mostrava.
> - **A inferência tem as mesmas 22 colunas do histórico, menos o `Churn`.** É o esperado: são os clientes que o
>   modelo vai pontuar, e a resposta deles não é conhecida.
> - **A visualização transposta** mostra uma linha por variável e já revela padrões:
>   - `SeniorCitizen` vem como 0/1, enquanto `Partner`, `Dependents` e os serviços vêm como Yes/No;
>   - quem não tem internet aparece como "No internet service" em seis colunas de serviço;
>   - o `CustomerFeedback` repete informações de outras colunas (tempo de casa, mensalidade, forma de pagamento);
>   - o cliente que cancelou escreve "I have decided to churn": **possível sinal de vazamento** (hipótese, investigada na seção 7.3 do notebook 03).
> - Os dois `assert` interrompem o notebook se algo mudar nos arquivos: é uma proteção para quem rodar depois.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. Em que momento o modelo será usado: antes de o cliente pedir o cancelamento, ou também depois?
> 2. O `CustomerFeedback` foi coletado antes ou depois do cancelamento? Quem pergunta aos organizadores?
> 3. Treinamos dois modelos, um **com** e outro **sem** o `CustomerFeedback`, e comparamos? Se o "com texto" ficar perto de 99%, como explicamos à banca?
> 4. Mesmo fora do modelo, usamos o texto para entender os **motivos** de saída?
> 5. O `Prediction.csv` deve ter **1.409 linhas** (clientes) ou 1.461 (linhas físicas, como no slide)?
> 6. Qual fonte única o grupo usa: os CSVs de `dados/` ou as tabelas `bronze_*` do catálogo?
>
> **Respostas e decisões do grupo:**
>
> - **1 a 4 (D9, 10/11):** o texto foi escrito depois do desfecho (seção 7.3 do 03). Fica fora do modelo oficial e da ABT; o modelo com texto entra só como contraexemplo, a partir da `gold_churn_texto`. Falta a organização confirmar quando o texto foi coletado.
> - **5 (D13, 10/11):** 1.409 linhas, uma por `customerID`; a confirmar com a organização.
> - **6 (D11, 10/11):** tabelas `bronze_*` em `workspace.terra_signal`.

In [ ]:
# 1.2 Clientes x linhas físicas: por que o arquivo parece ter mais clientes do que tem
contagem = []
for nome, df in [("history.csv", hist_raw), ("inference.csv", inf_raw)]:
    with open(f"{PASTA}/{nome}", encoding="utf-8") as f:
        fisicas = sum(1 for _ in f) - 1                            # menos o cabeçalho
    quebras = int(df["CustomerFeedback"].str.contains("\n").sum())
    contagem.append({"arquivo": nome, "clientes": len(df), "linhas_fisicas": fisicas,
                     "linhas_a_mais": fisicas - len(df), "comentarios_com_quebra": quebras})
contagem = pd.DataFrame(contagem)
display(contagem)

fig, ax = plt.subplots(figsize=(10, 3.4))
yy = np.arange(len(contagem))[::-1]
ax.barh(yy + 0.18, contagem["linhas_fisicas"], height=0.34, color=CINZA, label="linhas físicas do arquivo")
ax.barh(yy - 0.18, contagem["clientes"], height=0.34, color=COR, label="clientes (registros)")
for y0, (_, r) in zip(yy, contagem.iterrows()):
    ax.text(r["linhas_fisicas"] + 60, y0 + 0.18, f"{virg(r['linhas_fisicas'], 0)} linhas físicas no arquivo", va="center", fontsize=9,
            color="#666666")
    ax.text(r["clientes"] + 60, y0 - 0.18,
            f"{virg(r['clientes'], 0)} clientes  ·  {virg(r['linhas_a_mais'], 0)} linhas a mais, de "
            f"{r['comentarios_com_quebra']} comentários com quebra de linha", va="center", fontsize=9, color=COR,
            fontweight="bold")
ax.set_yticks(yy, contagem["arquivo"])
ax.set_xlim(0, contagem["linhas_fisicas"].max() * 1.9)
ax.set_xticks([])
ax.spines[["bottom", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
# sem legenda: o rótulo ao lado de cada barra já diz o que ela é (cinza = linhas físicas; azul = clientes)
fig.suptitle("O arquivo tem mais linhas do que clientes: alguns comentários do feedback ocupam várias linhas.\n"
             "Contar linhas daria 5.912 e 1.461; os clientes de verdade são 5.634 e 1.409",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["CustomerFeedback"], fonte="history.csv e inference.csv")
plt.show()

> **Como ler o gráfico 1.2**
> - **Duas barras por arquivo:** cinza = linhas físicas do arquivo (o que um editor de texto ou o `wc -l` contaria);
>   azul = clientes de verdade (registros que o pandas montou).
> - **O texto ao lado da barra azul** diz quantas linhas sobram e quantos comentários com quebra de linha as explicam.
> - **O que procurar:** a diferença entre as duas barras. Ela não é erro do arquivo: é o texto do feedback ocupando mais
>   de uma linha entre aspas. Quem contar linhas em vez de clientes vai achar que a base tem 5.912 clientes, como o slide.

## 2. Camada bronze: guardar o dado bruto

**O que faço:** gravo as duas bases no Unity Catalog exatamente como vieram, acrescentando só duas colunas
de controle: o arquivo de origem e a data e hora da carga.

**Por quê?**
- **Arquitetura medalhão** (bronze → silver → gold): cada camada tem um papel.
  - **Bronze:** o dado bruto, sem nenhuma alteração. É a "prova" do que recebi.
  - **Silver:** o dado limpo e tipado (notebook 02).
  - **Gold:** o dado pronto para o negócio e para o modelo (ABT e tabela de scores).
- **Rastreabilidade:** se uma regra de limpeza estiver errada, consigo refazer a partir da bronze sem voltar aos CSVs.
- **Colunas de controle** (`_origem`, `_carregado_em`): dizem de onde e quando o dado veio. O sublinhado no início
  indica que não são variáveis do negócio, e a limpeza as remove.
- `mode("overwrite")`: rodar de novo substitui a tabela, em vez de duplicar as linhas.

In [ ]:
# 2. Camada bronze: grava as bases brutas no Unity Catalog, com metadados de carga
from datetime import datetime, timezone

# CATALOG, SCHEMA e FQ vêm do _comum (um lugar só para o endereço do catálogo)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")

def para_bronze(df: pd.DataFrame, origem: str, tabela: str, descricao: str):
    df = df.copy()
    df["_origem"] = origem
    df["_carregado_em"] = datetime.now(timezone.utc).isoformat()
    (spark.createDataFrame(df).write.mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(f"{FQ}.{tabela}"))
    spark.sql(f"COMMENT ON TABLE {FQ}.{tabela} IS '{descricao}'")
    n = spark.table(f"{FQ}.{tabela}").count()
    print(f"{FQ}.{tabela}: {n:,} linhas".replace(",", "."))
    return n

n_hist = para_bronze(hist_raw, "history.csv", "bronze_history", "history.csv bruto: base de treino com o alvo Churn")
n_inf = para_bronze(inf_raw, "inference.csv", "bronze_inference", "inference.csv bruto: clientes a pontuar (sem Churn)")
assert (n_hist, n_inf) == (len(hist_raw), len(inf_raw)), "a tabela gravada tem número de linhas diferente do arquivo"

> **Explicação**
> - As duas tabelas ficam em **Catalog → workspace → terra_signal**, com 5.634 e 1.409 linhas.
> - Todas as colunas são gravadas como **texto (string)**, exatamente como lidas. A conversão de tipos fica para a silver.
> - O `assert` confere que nenhuma linha se perdeu ou duplicou na gravação.
> - Ao contrário das tabelas `workspace.default.history` e `inference` (5.773 e 1.435 linhas, com registros quebrados),
>   estas têm a contagem correta e são a **fonte oficial** para os próximos notebooks.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. O caminho `PASTA` encontra sozinho a pasta `dados/` do repositório. Todos deixam os CSVs em `dados/` na própria branch?
> 2. Todos têm permissão de escrita no schema `workspace.terra_signal`? Se não, quem pode dar acesso?
> 3. Usamos `terra_signal` como schema do grupo? Os notebooks de todos precisam apontar para o mesmo lugar.
> 4. Apagamos ou ignoramos as tabelas do upload em `workspace.default` (desalinhadas), para ninguém usar por engano?
>
> **Respostas e decisões do grupo:**
>
> - **3 (D11, 10/11):** o schema do grupo é `workspace.terra_signal`.
> - **4 (D11, 10/11):** ignorar as tabelas do upload em `workspace.default`, sem apagar.

## 3. Metadados: o que sei antes de olhar os dados

**O que faço:** registro, para cada coluna, o conhecimento de negócio e a classificação estatística,
**antes** de calcular qualquer coisa.

**Por quê?**
- **Separar o esperado do observado.** Primeiro declaro o que cada variável *deveria* ser (tipo, valores
  possíveis). Na seção 4, o notebook compara isso com os dados e aponta sozinho o que não confere.
  Assim, os problemas de qualidade são **encontrados**, e não digitados.
- **A classificação estatística define a análise.** O tipo da variável decide que estatística descritiva
  uso e que teste aplico contra o churn.

> **Resumo: tipos de variáveis**
> | Tipo | O que é | Descrever com | Testar contra o churn com |
> |---|---|---|---|
> | Qualitativa nominal | Categorias sem ordem (sexo, forma de pagamento) | Frequência e % | Qui-quadrado, V de Cramér |
> | Qualitativa ordinal | Categorias com ordem (contrato mensal < 1 ano < 2 anos) | Frequência e % | Qui-quadrado, tendência |
> | Qualitativa binária | Só duas categorias (sim/não) | Frequência e % | Qui-quadrado, razão de chances |
> | Quantitativa discreta | Contagem (meses de casa) | Média, mediana, quartis | Mann-Whitney |
> | Quantitativa contínua | Medida (valor em R$) | Média, mediana, quartis | Mann-Whitney |

**O que registro para cada coluna**

| Campo | Significado |
|---|---|
| `grupo` | De onde vem a informação: cliente, contrato, serviços, financeiro, texto, alvo |
| `tipo_estatistico` | A classificação acima |
| `papel` | `id` (identificador), `feature` (explicativa), `texto` (texto livre) ou `alvo` (Y) |
| `descricao` | O que a coluna significa para o negócio |
| `unidade` | Meses, R$ ou vazio |
| `dominio_esperado` | Lista de categorias aceitas ou intervalo numérico (mínimo, máximo) |

**Por que `SeniorCitizen` é qualitativa, se vem como 0 e 1?** O número é só um código de "sim" ou "não". Calcular
a média de `SeniorCitizen` dá a proporção de idosos, mas a variável não mede quantidade.

**O que verifiquei e o que confirmei.** A base não veio com dicionário oficial. Por isso, a coluna `confirmacao` registra
de onde vem cada descrição:
- **Verificado:** os próprios dados confirmam. Exemplo: "No internet service" só aparece para quem não tem internet.
- **Organizadores:** confirmado pelo Carlos (Databricks) em 10/11, depois das perguntas do grupo:
  - `SeniorCitizen`: 1 = idoso; sem data de nascimento, considerar 65 anos ou mais;
  - `MonthlyIncome`: renda **familiar** mensal, em R$, de uma base externa sem garantia de precisão; pode ser ignorada;
  - `tenure`: em meses, e 72 é um teto real (a empresa tem cerca de 7 anos), não um corte;
  - `MonthlyCharges`: valor do último mês faturado, com taxas e adicionais; para quem saiu, o último mês antes do cancelamento;
  - `TotalCharges`: total acumulado pago, a soma das mensalidades. O preço muda com plano e promoções, e a base não tem o
    histórico de preços: por isso `MonthlyCharges` × `tenure` é só uma aproximação;
  - `PhoneService`: `yes`/`no` em minúsculo é só erro de padronização, da mesma fonte.
- **Evidente pelo nome e pelos valores:** `gender` com Male/Female, `Contract` com os três tipos de contrato.

**A moeda é o real (R$), em todo o dataset.** O feedback escreve "$73.9", o que tinha nos levado a assumir dólar. O texto
parece seguir o formato da base pública de referência; vale a moeda informada pelos organizadores.

> Os metadados estão definidos no `_comum` (tabela `meta`), para todos os notebooks usarem os mesmos. Aqui, mostro o resultado.

In [ ]:
display(meta.reset_index())
display(meta["tipo_estatistico"].value_counts().rename("n_colunas").reset_index())
display(meta["confirmacao"].str.split(":").str[0].value_counts().rename("n_colunas").reset_index())

In [ ]:
# 3.1 Retrato do dicionário: tipos estatísticos das 23 colunas e de onde vem cada descrição
ORDEM_TIPOS = ["qualitativa nominal", "qualitativa binária", "qualitativa ordinal",   # agrupados por família
               "quantitativa contínua", "quantitativa discreta", "identificador", "texto livre"]
tipos = meta["tipo_estatistico"].value_counts().reindex(ORDEM_TIPOS).dropna().astype(int)
ROTULO_CONF = {"Verificado": "Verificado nos dados", "Organizadores": "Confirmado pelos organizadores",
               "Evidente pelo nome da coluna e pelos valores": "Evidente pelo nome", "PREMISSA": "Premissa a confirmar"}
origem = meta["confirmacao"].str.split(":").str[0].map(ROTULO_CONF)
conf = origem.value_counts().reindex(list(ROTULO_CONF.values())).dropna().astype(int)
cols_org = list(meta.index[origem.eq("Confirmado pelos organizadores")])
n_premissa = int(conf.get("Premissa a confirmar", 0))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4), gridspec_kw={"width_ratios": [1.2, 1.1]})
t = tipos[::-1]                                                   # a primeira família fica no topo
ax1.barh(t.index, t.values, color=[COR if "qualitativa" in i else DESTAQUE if "quantitativa" in i else CINZA
                                   for i in t.index], height=0.6)
for v, n in zip(t.values, t.index):
    ax1.text(v + 0.15, n, str(v), va="center", fontsize=9, color=TEXTO)
ax1.set_title("Tipo estatístico, por família", loc="left", fontsize=10.5)
c = conf[::-1]
ax2.barh(c.index, c.values, color=[DESTAQUE if "organizadores" in i or "Premissa" in i else COR if "Verificado" in i
                                   else CINZA for i in c.index], height=0.6)
for v, n in zip(c.values, c.index):
    extra = "\n" + textwrap.fill(", ".join(cols_org), 45) if "organizadores" in n else ""
    ax2.text(v + 0.2, n, f"{v}{extra}", va="center", fontsize=8.5, color=TEXTO)
ax2.set_title("De onde vem a descrição", loc="left", fontsize=10.5)
for ax in (ax1, ax2):
    ax.set_xticks([])
    ax.spines[["bottom", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
ax2.set_xlim(0, conf.max() * 2.6)
fim = "nenhuma premissa ficou em aberto" if n_premissa == 0 else f"{n_premissa} premissas ainda a confirmar"
fig.suptitle(f"Das 23 colunas, 16 explicativas são qualitativas e 4 quantitativas; {conf['Verificado nos dados']} descrições "
             f"foram verificadas nos dados,\n{conf['Confirmado pelos organizadores']} confirmadas pelos organizadores, e {fim}",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, cols_org)
plt.show()

> **Como ler o gráfico 3.1**
> - **À esquerda, o tipo estatístico de cada coluna** (seção 3), agrupado por família: de cima para baixo, azul =
>   qualitativas, laranja = quantitativas, cinza = identificador e texto livre. O tipo define que medida descritiva e que teste uso nas seções 5 a 7.
> - **À direita, de onde vem a descrição de cada coluna:** verificada nos próprios dados, confirmada pelos organizadores
>   (em laranja, com o nome das colunas), evidente pelo nome e pelos valores, ou premissa.
> - **O que procurar:** a base é majoritariamente qualitativa, então a maior parte das análises vai usar tabelas cruzadas,
>   qui-quadrado e V de Cramér. E nenhuma descrição depende mais de premissa: as dúvidas foram respondidas pelos organizadores.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. Quem confirma com os organizadores as duas premissas: o que define idoso em `SeniorCitizen` e a unidade de `MonthlyIncome`?
> 2. `MonthlyIncome` é contínua (valor monetário, como aqui) ou discreta (como em outro notebook do grupo)?
> 3. Os serviços ficam com 3 categorias, ou viram binários com a falta de internet só em `InternetService`? Isso muda a análise da seção 6.
> 4. O limite de 72 meses em `tenure` é o tempo máximo real da base ou um corte de quem montou os dados?
> 5. O gráfico 3.1 entra na apresentação ou fica só na documentação?
>
> **Respostas e decisões do grupo:**
>
> - **1 (Carlos, 10/11):** `SeniorCitizen` 1 = idoso, considerar 65 anos ou mais. `MonthlyIncome` é renda familiar mensal, em R$, de base externa sem garantia de precisão; pode ser ignorada (D12 do checklist).
> - **4 (Carlos, 10/11):** 72 meses é teto real: a empresa tem cerca de 7 anos.
> - Atenção: a moeda de todo o dataset é **R$**, não dólar.

## 4. Qualidade dos dados: o observado contra o esperado

**O que faço:** comparo cada valor da base com o domínio esperado declarado na seção 3 e verifico regras
de consistência entre colunas.

**Por quê?**
- Um modelo aprende o que está nos dados, inclusive os erros. Uma grafia diferente (`yes` e `Yes`) vira duas
  categorias; um `unknown` no meio de números impede o cálculo da média.
- **Dois tipos de verificação:**
  1. **Dentro de cada coluna (4.1):** o valor está no domínio? Categoria prevista, número dentro do intervalo.
  2. **Entre colunas (4.2):** as regras de negócio valem? Exemplo: quem não tem internet não pode ter segurança online.
- **Verifico histórico e inferência juntos.** Se a inferência tiver um problema que o histórico não tem,
  a limpeza precisa prever esse caso.

Não corrijo nada aqui: este notebook só **diagnostica**. As correções ficam na limpeza (notebook 02).

> A verificação está no `_comum` (tabela `qualidade`), porque a análise exploratória também usa o resultado.

In [ ]:
print("Colunas com algum problema:")
display(qualidade[qualidade["vazios"] + qualidade["fora_do_dominio"] > 0])
print(f"Colunas sem nenhum problema: {qualidade.loc[qualidade['pct_problema'] == 0, 'coluna'].nunique()} de {len(meta)}")

In [ ]:
# 4.1b Gráfico da qualidade: as três colunas com problema em detalhe e as 20 limpas resumidas
q_graf = (qualidade.assign(pct=100 * (qualidade["vazios"] + qualidade["fora_do_dominio"]) / qualidade["n"],
                           n_inv=qualidade["vazios"] + qualidade["fora_do_dominio"])
          .pivot(index="coluna", columns="base", values=["pct", "n_inv"]).reindex(meta.index))
tem_problema = q_graf["pct"].fillna(0).max(axis=1) > 0
prob = q_graf[tem_problema].sort_values(("pct", "historico"))       # o maior fica no topo do gráfico
limpas = list(q_graf.index[~tem_problema])
TIPO_PROBLEMA = {"tenure": "texto 'unknown' no lugar do número", "PhoneService": "grafia minúscula (yes / no)",
                 "TotalCharges": "texto 'null' no lugar do valor"}

fig, (ax, ax_txt) = plt.subplots(2, 1, figsize=(11, 6.2), gridspec_kw={"height_ratios": [2.4, 1]})
y = np.arange(len(prob))
for desl, base, cor, nome in [(0.2, "historico", DESTAQUE, "histórico"), (-0.2, "inferencia", COR, "inferência")]:
    ax.barh(y + desl, prob[("pct", base)], height=0.36, color=cor)
    for yy, col in zip(y, prob.index):
        pct, n_inv = prob.loc[col, ("pct", base)], int(prob.loc[col, ("n_inv", base)])
        ax.text(pct + 0.06, yy + desl, f"{nome}: {virg(pct, 1)}% ({virg(n_inv, 0)} clientes)",
                va="center", fontsize=8.5, color=cor, fontweight="bold")
ax.set_yticks(y, [f"{c}\n{TIPO_PROBLEMA[c]}" for c in prob.index], fontsize=9.5)
ax.set_xlim(0, 7.5)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{virg(v, 0)}%"))
ax.set_xlabel("% de valores vazios ou fora do domínio esperado")
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
ax.set_title("As 3 colunas com problema", loc="left", fontsize=10.5)

ax_txt.axis("off")                                                 # as 20 colunas limpas, em texto, sem barras vazias
ax_txt.text(0, 0.95, f"As outras {len(limpas)} colunas não têm nenhum valor inválido nas duas bases:",
            fontsize=10.5, fontweight="bold", va="top", transform=ax_txt.transAxes)
ax_txt.text(0, 0.68, textwrap.fill(", ".join(limpas) + " (só no histórico).", width=120),
            fontsize=9, color="#777777", va="top", transform=ax_txt.transAxes)

fig.suptitle(f"{len(limpas)} das {len(q_graf)} colunas estão limpas; os problemas se concentram em três, nenhuma passa\n"
             "de 5%, e aparecem na mesma proporção no histórico e na inferência",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, list(prob.index)[::-1], fonte="history.csv e inference.csv")
plt.show()

> **Como ler o gráfico 4.1b**
> - **Parte de cima, as 3 colunas com problema:** cada uma tem duas barras, laranja para o histórico e azul para a
>   inferência. O comprimento é o % de valores vazios ou fora do domínio esperado (seção 3); ao lado, o % e o número de
>   clientes afetados. Embaixo do nome da coluna, o tipo de problema.
> - **Parte de baixo, as 20 colunas limpas:** listadas em texto, sem barras, porque não há nada a mostrar nelas.
> - **O que procurar:**
>   - uma barra muito maior numa base do que na outra indicaria um problema que a limpeza do histórico não prevê.
>     Não é o caso: as proporções são parecidas (`TotalCharges` 4,9% x 4,5%; `PhoneService` 1,2% x 1,6%; `tenure` 0,3% x 0,3%);
>   - nenhuma coluna passa de 5%: os problemas são pontuais e não comprometem as variáveis.

In [ ]:
# 4.2 Regras de consistência entre colunas (o que o negócio diz que precisa valer junto)
def checar_regras(df: pd.DataFrame) -> dict:
    serv = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
    sem_internet = df["InternetService"].eq("No")
    sem_fone = df["PhoneService"].str.lower().eq("no")
    tn = pd.to_numeric(df["tenure"], errors="coerce")
    mc = pd.to_numeric(df["MonthlyCharges"], errors="coerce")
    tc = pd.to_numeric(df["TotalCharges"], errors="coerce")
    razao = tc / (mc * tn)
    texto = df["CustomerFeedback"].str.lower().str.split().str.join(" ")   # ignora caixa e espaços
    return {
        "ID fora do padrão 0000-XXXXX": int((~df["customerID"].str.match(r"^\d{4}-[A-Z]{5}$")).sum()),
        "ID duplicado": int(df["customerID"].duplicated().sum()),
        "Linha inteira duplicada": int(df.duplicated().sum()),
        "Feedback repetido na mesma base": int(texto.duplicated().sum()),
        "Sem internet, mas com serviço de internet": int((sem_internet & df[serv].ne("No internet service").any(axis=1)).sum()),
        "Com internet, mas serviço marcado 'No internet service'": int((~sem_internet & df[serv].eq("No internet service").any(axis=1)).sum()),
        "Sem telefonia, mas MultipleLines diferente de 'No phone service'": int((sem_fone & df["MultipleLines"].ne("No phone service")).sum()),
        "TotalCharges menor que MonthlyCharges (tenure > 1)": int(((tc < mc) & (tn > 1)).sum()),
        "TotalCharges fora de ±20% de MonthlyCharges × tenure": int(((razao < 0.8) | (razao > 1.2)).sum()),
        "tenure e TotalCharges inválidos ao mesmo tempo": int((tn.isna() & tc.isna()).sum()),
        "tenure = 0 com TotalCharges inválido": int(((tn == 0) & tc.isna()).sum()),
        "CustomerFeedback com quebra de linha": int(df["CustomerFeedback"].str.contains("\n").sum()),
    }

regras = pd.DataFrame({"historico": checar_regras(hist_raw), "inferencia": checar_regras(inf_raw)})
display(regras.reset_index(names="regra"))
print("IDs em comum entre histórico e inferência:", len(set(hist_raw["customerID"]) & set(inf_raw["customerID"])))
norm = lambda s: set(s.str.lower().str.split().str.join(" "))
print("Feedbacks iguais nas duas bases:", len(norm(hist_raw["CustomerFeedback"]) & norm(inf_raw["CustomerFeedback"])))

In [ ]:
# 4.2b Gráfico das regras de consistência: % de clientes que violam cada regra, nas duas bases
# Em %, porque o histórico tem 4 vezes mais clientes que a inferência; em número, a barra do histórico sempre seria maior
pct_txt = lambda v: "0%" if v == 0 else f"{virg(v, 2 if v < 0.1 else 1)}%"   # 0,02% em vez de 0,0%
N_BASE = {"historico": len(hist_raw), "inferencia": len(inf_raw)}
r = regras.copy()
for b in N_BASE:
    r[f"pct_{b}"] = 100 * r[b] / N_BASE[b]
r["total"] = r["historico"] + r["inferencia"]
r = r.sort_values(["pct_historico", "total"])
fig, ax = plt.subplots(figsize=(11, 6))
yy = np.arange(len(r))
for desl, base, cor in [(0.2, "historico", DESTAQUE), (-0.2, "inferencia", COR)]:
    ax.barh(yy + desl, r[f"pct_{base}"], height=0.36, color=[cor if v > 0 else CINZA for v in r[base]])
for y0, (regra, row) in zip(yy, r.iterrows()):
    if row["total"] == 0:
        ax.text(0.01, y0, "0 nas duas bases: regra respeitada", va="center", fontsize=8.5, color="#888888")
    else:
        ax.text(max(row["pct_historico"], row["pct_inferencia"]) + 0.03, y0,
                f"histórico {pct_txt(row['pct_historico'])} ({virg(row['historico'], 0)}) · "
                f"inferência {pct_txt(row['pct_inferencia'])} ({virg(row['inferencia'], 0)})",
                va="center", fontsize=8.5, color=TEXTO, fontweight="bold")
ax.set_yticks(yy, r.index, fontsize=9)
for rotulo, tot in zip(ax.get_yticklabels(), r["total"]):
    rotulo.set_color(TEXTO if tot > 0 else "#999999")
ax.set_xlim(0, r[["pct_historico", "pct_inferencia"]].max().max() * 2.1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{virg(v, 1)}%"))
ax.set_xlabel("% de clientes da base que não seguem a regra")
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=DESTAQUE, label=f"histórico ({virg(N_BASE['historico'], 0)} clientes)"),
                   Patch(color=COR, label=f"inferência ({virg(N_BASE['inferencia'], 0)} clientes)")],
          loc="lower right", frameon=False, fontsize=9)
n_ok = int((r["total"] == 0).sum())
fig.suptitle(f"{n_ok} das {len(r)} regras são respeitadas por todos os clientes; as exceções não passam de 1,5%\n"
             "e aparecem em proporções parecidas no histórico e na inferência",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["TotalCharges", "MonthlyCharges", "tenure", "CustomerFeedback", "MultipleLines", "InternetService"],
       fonte="history.csv e inference.csv")
plt.show()

> **Como ler o gráfico 4.2b**
> - **Cada linha é uma regra de negócio** que precisa valer para todos os clientes (por exemplo, "quem não tem internet não
>   pode ter segurança online").
> - **Duas barras por regra:** laranja = histórico; azul = inferência. O comprimento é o **% de clientes da base** que não
>   seguem a regra; ao lado, o % e o número de clientes. Uso % porque o histórico tem 4 vezes mais clientes que a
>   inferência: em número, a barra laranja seria sempre maior, mesmo com a mesma proporção.
> - **Linhas em cinza, sem barra:** regras respeitadas por todos os clientes nas duas bases.
> - **O que procurar:** regras com barra são exceções a explicar. Explicações possíveis (hipóteses, não confirmadas):
>   - quebras de linha no feedback (1,5% e 1,1%): formatação do texto, que explica as linhas físicas a mais da seção 1;
>   - `TotalCharges` fora de ±20% de mensalidade × tempo de casa (0,8% e 0,6%): talvez mudança de plano ao longo do tempo;
>   - `tenure` = 0 com `TotalCharges` nulo: clientes que acabaram de entrar e ainda não tiveram cobrança;
>   - `tenure` e `TotalCharges` inválidos ao mesmo tempo: um único cliente, o caso sem recuperação da 4.1.

> **Explicação (4.1 e 4.2)**
>
> **Diagnóstico:** 20 das 23 colunas estão limpas nas duas bases. Os problemas estão em três colunas:
>
> | Coluna | Problema | Histórico | Inferência | Tratamento decidido (10/11) |
> |---|---|---|---|---|
> | `tenure` | Texto `unknown` no lugar do número | 19 (0,3%) | 4 | Recuperar por `TotalCharges` ÷ `MonthlyCharges`, arredondado; se não der, mediana |
> | `PhoneService` | Grafia minúscula `yes` / `no` | 70 (1,2%) | 23 | Padronizar para `Yes` / `No` |
> | `TotalCharges` | Texto `null` | 278 (4,9%) | 64 | Preencher com `MonthlyCharges` × `tenure` |
>
> **Regras entre colunas:**
> - Todos os IDs seguem o padrão `0000-XXXXX`. Nenhum ID duplicado, nenhuma linha duplicada e nenhum cliente
>   presente nas duas bases. Nenhum comentário se repete, nem dentro de uma base nem entre as duas.
> - Nenhuma inconsistência de serviços: quem não tem internet está como "No internet service" em todos os seis
>   serviços, e quem não tem telefonia está como "No phone service" em `MultipleLines`.
> - `TotalCharges` fica dentro de ±20% de `MonthlyCharges` × `tenure` para quase todos (47 exceções no histórico e
>   9 na inferência). Isso dá base para o preenchimento acima (D2), que o Carlos considerou razoável como aproximação. As exceções
>   são mudança de plano e promoções (Carlos, 10/11).
> - Só 1 cliente tem `tenure` e `TotalCharges` inválidos ao mesmo tempo; nos outros, uma coluna recupera a outra.
> - 85 comentários no histórico (15 na inferência) têm quebra de linha: é o que explica a diferença entre
>   linhas físicas e clientes na seção 1.
>
> **Para o negócio:** a base é boa. Os problemas são pontuais, previsíveis e têm tratamento simples.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. `tenure` e `TotalCharges` inválidos: preenchemos pela relação mensalidade × meses, excluímos os clientes ou deixamos ausente para o modelo tratar?
> 2. Os 64 clientes da inferência sem `TotalCharges` **precisam** de previsão: qual tratamento serve para eles?
> 3. `TotalCharges` nulo com `tenure` = 0 vira 0?
> 4. O único cliente sem `tenure` e sem `TotalCharges`: excluir ou preencher (por exemplo, com a mediana)?
> 5. Os 47 clientes fora de ±20%: tratamos como mudança de plano (mantém) ou investigamos?
> 6. A grafia minúscula de `PhoneService` pode indicar outra fonte de dados? Vale perguntar aos organizadores?
> 7. Removemos as quebras de linha do feedback, mesmo que o texto não entre no modelo?
> 8. Há outra regra de negócio para testar (fatura digital × forma de pagamento, contrato × tempo de casa)?
>
> **Respostas e decisões do grupo:**
>
> - **1 (Carlos, 10/11):** preencher por mensalidade × meses é "razoável, desde que lembrem que é uma aproximação".
> - **5 (Carlos, 10/11):** as exceções vêm de mudança de plano e de pacotes promocionais; a base não tem o histórico de preços.
> - **6 (Carlos, 10/11):** só erro de padronização, mesma fonte. Pode tratar `yes`/`no` como `Yes`/`No`.

In [ ]:
# 4.3 A ausência é informativa? Churn de quem tem TotalCharges ou tenure ausente
from scipy.stats import chi2_contingency

y_h = hist_raw["Churn"].eq("Yes").astype(int)
tn_h = pd.to_numeric(hist_raw["tenure"], errors="coerce")
tc_h = pd.to_numeric(hist_raw["TotalCharges"], errors="coerce")

linhas_aus = []
for nome, aus in [("TotalCharges ausente", tc_h.isna()), ("tenure ausente", tn_h.isna())]:
    p = chi2_contingency(pd.crosstab(aus, y_h))[1]
    linhas_aus.append({"indicador": nome, "clientes": int(aus.sum()),
                       "churn_ausentes_%": round(100 * y_h[aus].mean(), 1),
                       "churn_demais_%": round(100 * y_h[~aus].mean(), 1),
                       "p_valor": round(p, 3)})
display(pd.DataFrame(linhas_aus))

# a diferença se mantém dentro do mesmo contrato e da mesma faixa de tempo de casa?
situacao = tc_h.isna().map({True: "TotalCharges ausente", False: "preenchido"})
faixa = pd.cut(tn_h, [-1, 6, 24, 48, 72], labels=["0-6 meses", "7-24", "25-48", "49-72"])
print("Taxa de churn (%) por contrato:")
display((100 * pd.crosstab(hist_raw["Contract"], situacao, values=y_h, aggfunc="mean")).round(1))
print("Taxa de churn (%) por faixa de tempo de casa:")
display((100 * pd.crosstab(faixa, situacao, values=y_h, aggfunc="mean")).round(1))

> **Explicação (4.3): a ausência também é informação**
>
> **Teoria aplicada:** um valor ausente pode ser só um defeito de carga (ausência ao acaso) ou pode estar ligado ao
> que quero prever. Para saber, cruzo "está ausente" (sim/não) com o churn e aplico o qui-quadrado de independência.
> Se a ausência se liga ao churn, imputar e esquecer joga essa informação fora; um indicador de ausência a preserva.
>
> **Como ler as tabelas:** a primeira compara a taxa de churn de quem tem o valor ausente com a dos demais. As duas
> seguintes repetem a comparação dentro do mesmo contrato e da mesma faixa de tempo de casa: se a diferença some
> dentro dos grupos, ela era só o perfil desses clientes; se continua, a ausência traz algo a mais.
>
> **O que encontrei:**
> - `TotalCharges` ausente: 33,1% de churn contra 26,4% nos demais (p = 0,016). A diferença se mantém no contrato
>   mensal (49,1% contra 42,3%) e no anual (19,2% contra 11,3%) e nas faixas acima de 6 meses de casa; só some no
>   bienal e nos primeiros 6 meses, onde as taxas já são extremas.
> - `tenure` ausente: 42,1% contra 26,6%, mas com só 19 clientes o teste não tem poder (p = 0,21).
> - Os 10 clientes com `tenure` = 0 estão todos entre os `TotalCharges` ausentes: são clientes novos, e o valor
>   provável para eles é 0, que é o que a regra mensalidade × tempo de casa dá.
>
> **Decisão (10/11):** a ausência de `TotalCharges` parece informativa, então os 278 clientes **não** são excluídos.
> O valor é preenchido pela regra (D2 e D3) e o indicador `fl_total_charges_ausente` guarda o sinal (D5); o modelo
> testa com e sem ele. Para `tenure`, com só 19 casos e p = 0,21, não há indicador (D6).

> **Anotações [André]: dúvidas para o grupo**
>
> 1. Criamos `fl_total_charges_ausente` (1 = estava ausente) e testamos no modelo?
> 2. Vale criar o indicador para `tenure`, com só 19 casos?
> 3. Se a ausência é informativa, excluir os 278 clientes do histórico ainda faz sentido?
> 4. Por que esses clientes não têm `TotalCharges`? Perguntar aos organizadores se é falha de sistema ou um tipo de cliente.
>
> **Respostas e decisões do grupo:**
>
> - **1 (D5):** criado; o modelo testa com e sem.
> - **2 (D6):** não criar.
> - **3 (D2):** não excluir; preencher pela regra.
> - **4:** ainda sem resposta da organização.

> **Resumo da seção 4**
>
> **O que encontrei:** 20 das 23 colunas estão limpas. Os problemas estão em `tenure` (19 `unknown`), `PhoneService`
> (70 com grafia minúscula) e `TotalCharges` (278 `null`). As regras entre colunas valem para todos os clientes, e
> `TotalCharges` ≈ `MonthlyCharges` × `tenure` vale para 99% deles. A ausência de `TotalCharges` parece não ser ao
> acaso: esses clientes saem mais (33,1% contra 26,4%, 4.3).
>
> **Decisões de limpeza (10/11):** no checklist de decisões abaixo (D1 a D8), o mesmo que abre o notebook 02.

## Checklist de decisões

O que este notebook levantou e o grupo decidiu em 10/11. **A mesma tabela abre o notebook 02**, para comparar o que foi
proposto aqui com o que a preparação aplica.

| # | Decisão | Opções | Proposta | Impacto | Evidência no 01 | No 02 | Depende de | Decisão do grupo |
|---|---|---|---|---|---|---|---|---|
| D1 | Grafia de `PhoneService` (70 + 23) | Padronizar | Padronizar para `Yes`/`No` | Sem isso, `yes` e `Yes` viram categorias diferentes no modelo | 4.1 | Aplicado | Carlos: só erro de padronização (respondido) | ☑ Padronizar para `Yes`/`No` (10/11) |
| D2 | `TotalCharges` nulo (278 + 64) | a) preencher com `MonthlyCharges` × `tenure`; b) excluir; c) deixar ausente | a | Excluir não serve para os 64 da inferência, que precisam de previsão, e perde o sinal da 4.3. A regra vale para 99% dos clientes | 4.1, 4.2, 4.3 | Aplicado | G2; Carlos: preencher assim é "razoável, desde que lembrem que é uma aproximação" (respondido); falta saber por que o valor está ausente | ☑ Preencher com `MonthlyCharges` × `tenure`, como aproximação (10/11) |
| D3 | `TotalCharges` nulo com `tenure` = 0 (10 + 1) | a) 0; b) outro valor | a (sai da própria regra: mensalidade × 0) | Clientes novos, sem cobrança ainda; efeito pequeno | 4.2 | Aplicado | G2 | ☑ 0 (10/11) |
| D4 | `tenure` = `unknown` (19 + 4) | a) `TotalCharges` ÷ `MonthlyCharges`, arredondado; b) mediana; c) excluir | a; mediana do histórico no único cliente sem as duas colunas | `tenure` é das variáveis mais ligadas ao churn: um erro aqui muda a faixa de tempo de casa do cliente | 4.1, 4.2 | Aplicado | G2; Carlos: 72 meses é teto real (respondido) | ☑ `TotalCharges` ÷ `MonthlyCharges`, arredondado e limitado a 72; mediana do histórico no caso sem os dois (10/11) |
| D5 | Indicador `fl_total_charges_ausente` | Criar ou não | Criar e comparar o modelo com e sem ele | Preserva o sinal da ausência (33,1% contra 26,4% de churn, p = 0,016) | 4.3 | Criado | Carlos: por que falta `TotalCharges` | ☑ Criar; o modelo testa com e sem (10/11) |
| D6 | Indicador `fl_tenure_ausente` | Criar ou não | Criar, mas tende a sair na seleção (só 19 casos, p = 0,21) | Pouca informação; uma coluna a mais | 4.3 | Não criado | G2 | ☑ Não criar: 19 casos, p = 0,21 (10/11) |
| D7 | `TotalCharges` fora de ±20% da regra (47 + 9) | a) manter; b) investigar; c) excluir | a | Mostra que o preenchimento de D2 é uma aproximação | 4.2 | Mantidos | Carlos: o preço muda com plano e promoções, e não há histórico de preços (respondido) | ☑ Manter (10/11) |
| D8 | Quebras de linha no `CustomerFeedback` (85 + 15) | Remover ou manter | Remover, se o texto for usado | Só importa se o texto entrar na análise ou no modelo | 4.2 | Aplicado | D9 | ☑ Trocar a quebra de linha por espaço (10/11) |
| D9 | `CustomerFeedback` no modelo | a) com; b) sem; c) os dois | c, com o modelo sem texto como principal | Possível vazamento: uma regra sobre o texto acerta 98,9% | Célula 2 (seção 1); 7.3 do 03 | Fora do 02 (modelo) | G1; Carlos: feedback antes ou depois do cancelamento | ☑ Fora da ABT. O texto vai na tabela separada `gold_churn_texto`, para o modelo com texto (contraexemplo); o modelo oficial é sem texto (10/11) |
| D10 | Serviços com 3 categorias ou sim/não | a) manter 3; b) binário | Testar as duas | "No internet service" repete `InternetService` em 6 colunas: redundância | 3 | Fora do 02 (notebook 04) | G4 | ◐ Sugestão: serviços como sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; seleção final na modelagem (10/11) |
| D11 | Fonte e schema do grupo | a) CSVs de `dados/`; b) tabelas `bronze_*` em `terra_signal` | b | Todos os notebooks; os números precisam bater entre os integrantes | 1.2, 2 | Aplicado: o 02 lê das tabelas `bronze_*` | G3; Carlos: CSV com quebra de linha, schema | ☑ Tabelas `bronze_*` em `workspace.terra_signal`; ignorar (sem apagar) as do upload em `workspace.default` (10/11) |
| D12 | `MonthlyIncome` no modelo | a) manter como candidata; b) ignorar | a, e deixar a seleção decidir | Base externa, sem garantia de precisão; os organizadores dizem que pode ser ignorada. Na exploratória, a renda não muda o risco | 3; seção 10 do 03 | Convertida para número e mantida | Carlos: renda familiar em R$, base externa (respondido) | ◐ Sugestão: descartar (IV abaixo de 0,02, base externa). Fica na ABT; seleção final na modelagem (10/11) |
| D13 | Linhas do `Prediction.csv` | a) 1.409 (clientes); b) 1.461 (linhas físicas) | a | Arquivo com o número errado de linhas pode invalidar a entrega | 1.2 | Fora do 02 (entrega) | Organização: confirmar | ☑ 1.409, uma por `customerID`; a confirmar com a organização (10/11) |
| D14 | Formato da gold | — | `gold_churn_treino` (5.634, com `Churn` 1/0) e `gold_churn_inferencia` (1.409, sem `Churn`), mesmas colunas e ordem; números como número, categorias como texto; comentário em cada coluna; `gold_churn_previsoes` depois do modelo | É o que a modelagem usa; formato estável evita retrabalho | — | Fora do 02 (notebook 04) | — | ☑ ABT com todas as candidatas (originais limpas, serviços nas duas versões, indicador e derivadas) + dicionário com a sugestão de cada coluna; texto em `gold_churn_texto` (10/11) |
| D15 | Outras variáveis sem poder preditivo | a) descartar já; b) deixar a seleção decidir | a: `gender`, `PhoneService` e `MultipleLines` | IV abaixo de 0,02; modelo mais simples de explicar; sexo é variável sensível | Seção 7 e 8 do 03 | Fora do 02 (notebook 04) | — | ◐ Sugestão: descartar `gender`, `PhoneService` e `MultipleLines` (IV abaixo de 0,02). Ficam na ABT; seleção final na modelagem (10/11) |
| D16 | `MonthlyCharges` e `TotalCharges` (repetem outras variáveis) | a) manter; b) trocar por derivada; c) deixar a seleção decidir | Decidir no 04, com as derivadas | Repetem `InternetService` e `tenure` (associação ≥ 0,8) | Seção 6 e 8 do 03 | Fora do 02 (notebook 04) | 04 | ◐ Sugestão: manter `MonthlyCharges` (ou a mensalidade relativa ao pacote) e descartar `TotalCharges` (repete `tenure`, ρ 0,89). As três ficam na ABT; seleção final na modelagem (10/11) |
| D17 | Fonte do 03 | a) CSVs; b) tabelas `bronze_*` | b | Coerência com a D11; os números não mudam | — | Fora do 02 (notebook 03) | — | ☑ Ler da bronze (10/11) |
| D18 | Divisão de papéis | — | A preparação entrega a ABT e sugere; a seleção final das variáveis é feita na modelagem | Quem decide a seleção final das variáveis | — | — | — | ☑ Como proposto (10/11) |
| D19 | Seleção de variáveis | a) só as recomendadas; b) recomendadas + seleção para frente por AUC; c) testes estatísticos com correção para múltiplos testes e retirada das duplicatas | c, conferida com b | É o que entra no modelo | — | — | Notebook 05 | ☑ As 41 colunas do notebook 05 (seções 2 e 3): saem 6 categóricas redundantes e 12 variáveis (duplicatas exatas, versões repetidas do tempo de casa, `gender` sem associação após a correção FDR). A conferência com os métodos da aula (11 variáveis, AUC 0,841) cabe dentro das 41 (11/11) |
| D20 | Segmentação de clientes | a) sem segmentação; b) K-means no comportamento, sem o churn, cruzado com risco × valor | b | Diz que tipo de cliente está em risco e que oferta faz sentido | — | — | Notebook 07 | ◐ Sugestão para o negócio e a apresentação (11/11) |
| D21 | Modelo final | a) logística simples; b) regularização (Lasso, Ridge, Elastic Net); c) árvores (*random forest*, XGBoost, CatBoost) | O de melhor desempenho e mais consistente entre treino, teste e base total | É o modelo que vai para a entrega | — | — | Notebook 06 | ☑ Notebook 06 (modelo do grupo): logística Lasso, 41 colunas, C = 0,3, classes balanceadas, vencedora pela regra de consistência (Ridge a 0,003); AUC 0,830 no teste. Atenção: o balanceamento infla a probabilidade (média de 41% para 27% de churn real) (11/11) |
| D22 | Ponto de corte do Sim/Não | a) 0,5; b) o que maximiza o F1; c) pelo custo da ação | b | Define quantos clientes viram "Yes" | — | — | Notebook 06 | ☑ 0,5046: maior F1 nas probabilidades fora da dobra do treino (notebook 06) (11/11) |
| D23 | Arquivo da entrega | — | 1.409 linhas, `customerID` e `prediction` (Yes/No), na ordem do `inference.csv`, no formato do notebook oficial | Arquivo fora do formato pode invalidar a entrega | — | — | D13; notebook 08 | ☑ `prediction_grupo4_<data_hora>.csv` gerado no 08 e conferido nas 8 regras: 571 de 1.409 com Yes (40,5%); falta comparar os nomes das colunas com o notebook oficial (11/11) |
| D24 | Monitoramento do modelo | a) sem monitoramento; b) PSI das variáveis e das notas, taxa de Yes e reprodução pelo MLflow, com semáforo e app | b | Avisa quando o modelo envelhece, antes de o resultado de negócio mostrar | — | — | Notebook 09 | ◐ Proposta: notebook 09 e app em Streamlit (11/11) |
| D25 | Motor de atendimento | a) só a fila de retenção; b) a nota, os motivos e a oferta de cada cliente na hora da ligação, e a comparação de pacotes para quem ainda não é cliente | b | Leva o modelo para quem fala com o cliente | — | — | Notebook 10 | ◐ Proposta: notebook 10 e página Atendimento do app (12/11) |

**Como ler:** "Evidência no 01" aponta a seção do notebook 01. "No 02" diz se o notebook 02 já aplica a proposta.
"Depende de" liga às perguntas para o grupo (G1 a G4) e ao Carlos. ☑ = decidido (com a data); ◐ = sugestão nossa, a confirmar (as de variáveis, na modelagem); ☐ = em aberto.

## Resumo do notebook 01

| Etapa | O que fiz | O que encontrei |
|---|---|---|
| **1. Leitura** | Li os CSVs como texto, sem converter nada | **5.634** clientes no histórico e **1.409** na inferência. Os arquivos têm 5.912 e 1.461 linhas físicas porque o feedback tem quebras de linha dentro do texto |
| **2. Bronze** | Gravei as bases como chegaram, com origem e data de carga | Tabelas `bronze_history` e `bronze_inference` no Unity Catalog |
| **3. Metadados** | Descrevi cada coluna: grupo, tipo estatístico, papel, domínio esperado e o que foi verificado ou assumido | 23 colunas: 1 identificador, 20 explicativas, 1 texto livre e o alvo `Churn`. Moeda em R$. Nenhuma premissa em aberto: as dúvidas sobre `SeniorCitizen`, `MonthlyIncome`, `tenure`, `MonthlyCharges` e `TotalCharges` foram respondidas pelos organizadores (10/11) |
| **4. Qualidade** | Comparei cada valor com o domínio esperado e chequei regras entre colunas | 20 de 23 colunas sem problemas. Problemas só em `tenure` (19 `unknown`), `PhoneService` (grafia "yes"/"no") e `TotalCharges` (278 `null`). A ausência de `TotalCharges` se liga ao churn (33,1% contra 26,4%): indicador `fl_total_charges_ausente` criado (D5) |

**Vazamento:** o `CustomerFeedback` conta o desfecho ("I have decided to churn"). Fica fora do modelo oficial e da ABT (D9); o modelo com texto entra só como contraexemplo.

**Próximo passo:** notebook 02, que aplica as decisões D1 a D8 e grava as tabelas limpas.